In [ ]:
!pip install torchsummary
!pip install torchinfo

## Tiny Yolo

In [ ]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimov2

    # Get the YOLO model
    model = tinysimov2.yolo_v8_s()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


## V3 Fused weight

### Before fusuion


In [1]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimov3

    # Get the YOLO model
    model = tinysimov3.yolo_v8_s()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


### After fusion

In [2]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimov3

    # Get the YOLO model
    model = tinysimov3.yolo_v8_s()
    model.fuse()  # Critical step for weight fusion!
    model.eval()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary_fused.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


/home/mdi220/anaconda3/envs/YOLO/lib/python3.10/site-packages/torch/nn/modules/module.py:935: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647327489/work/build/aten/src/ATen/core/TensorBody.h:489.)
  param_grad = param.grad


In [ ]:
import torch
from torchsummary import summary
from nets import tinysimov2

model = tinysimov2.yolo_v8_s()

In [ ]:
print(model)